# Modelleren van patiëntervaringsscores per zorginstelling en specialisme met PROC FACTMAC


## Managementsamenvatting

Een multisite zorgsysteem wil de **interactiestructuur** leren kennen tussen
zorginstellingen en klinische specialismen die patiënttevredenheidsscores
bepaalt, zodat het combinaties van instelling/specialisme kan opsporen die
onder- of overpresteren. Dit notebook past een factorisatiemachine toe met
**PROC FACTMAC**, waarbij `facility` en `specialty` worden behandeld als de
twee nominale kenmerkvelden en de tevredenheidsscore van 1-10 als het
intervaldoel, en evalueert vervolgens de gereconstrueerde scores.

Op de 100 gesimuleerde contacten behaalt het model een **trainings-R-kwadraat
van 0,516** (gemiddelde absolute fout 0,95 scorepunten, RASE 1,25), en de
voorspelde celgemiddelden scheiden duidelijk de sterkste en zwakste
combinaties — `Noordkliniek`/`Cardiologie` bovenaan tegenover
`Zuidkliniek`/`Cardiologie` onderaan — waarbij de ingebedde interactie wordt
teruggevonden in plaats van in te storten naar het algemene gemiddelde van
ongeveer 6,8.


## Gegevensbronnen

Alle gegevens worden inline gegenereerd door een DATA-stap
(`call streaminit(20240531)` + `rand()`), dus het notebook is volledig
zelfstandig — geen externe bestanden of netwerktoegang. Deze omgeving draait
zonder licentie, wat de uitvoer beperkt tot 100 observaties, dus het ontwerp
is zo gedimensioneerd dat de factorisatiemachine **binnen 100 contacten**
kan worden gedemonstreerd: drie instellingen gekruist met twee specialismen
(zes cellen, gemiddeld ongeveer 17 contacten elk — genoeg signaal per cel
zodat stochastische gradiëntafdaling de structuur kan terugvinden).

**WORK.ENCOUNTERS** — 100 synthetische patiëntcontacten (één rij per contact).

| Variable | Type | Rol | Beschrijving |
|----------|------|------|-------------|
| `facility` | char(12) | Invoer (nominaal) | Zorglocatie — `Noordkliniek`, `CentraalMed` of `Zuidkliniek` |
| `specialty` | char(14) | Invoer (nominaal) | Klinisch specialisme — `Cardiologie` of `Oncologie` |
| `satisfaction` | num | Doel (interval) | Patiëntervaringsscore op een schaal van 1-10, gegenereerd uit een instellingsbias + specialismebias + een echte interactieterm tussen instelling×specialisme + Gaussiaanse ruis, vervolgens afgekapt tot [1,10] en afgerond op 0,1 |

Het onderliggende ontwerp bevat goed gescheiden biases per instelling en per
specialisme plus een sterk interactie-effect, zodat een factorisatiemachine
structuur zou moeten terugvinden die een gemiddelde op alleen instelling of
alleen specialisme zou missen.


# Modelleren van patiëntervaringsscores met PROC FACTMAC

Multisite zorgsystemen verzamelen patiënttevredenheidsscores over veel
**zorginstellingen** en klinische **specialismen**. Gemiddelde scores per
instelling of per specialisme alleen verbergen het interessante signaal: een
specialisme kan op de ene locatie uitblinken en op een andere moeite hebben.
Een **factorisatiemachine** vangt precies dit soort paarsgewijze interactie
door latente factoren te leren voor elke instelling en elk specialisme, en
modelleert vervolgens elke score als een globaal gemiddelde plus
per-kenmerk-effecten plus hun interactie.

`PROC FACTMAC` past dit model toe met stochastische gradiëntafdaling. In dit
notebook gaan we:

1. Een synthetische contacttabel genereren met een ingebedde interactie
   instelling x specialisme, gedimensioneerd voor de omgeving met 100
   observaties.
2. Een factorisatiemachine fitten met `PROC FACTMAC`, waarbij gescoorde
   voorspellingen en een iteratiegeschiedenis worden opgevraagd.
3. De reconstructiefout evalueren en de combinaties instelling x specialisme
   naar boven halen die het model als sterkst en zwakst markeert.


## Stap 1 - Genereer synthetische patiëntervaringsgegevens

We simuleren 100 contacten over 3 instellingen en 2 specialismen. Elke
instelling en elk specialisme draagt een verborgen bias, en we voegen een
echte **interactie**term toe zodat bepaalde combinaties van
instelling/specialisme hoger of lager scoren dan hun individuele biases
zouden voorspellen. Gaussiaanse ruis (standaarddeviatie 0,25) maakt de
scores realistisch, en we kappen af op de schaal 1-10 en ronden af op één
decimaal. De seed van `call streaminit` maakt de gegevens reproduceerbaar.


In [1]:
GEGEVENS encounters;
    CALL streaminit(20240531);
    LENGTE facility $12 specialty $14;

    /* Benoemde instellingen en klinische zorglijnen */
    REEKS facs[3] $12 _temporary_ (
        'Noordkliniek' 'CentraalMed' 'Zuidkliniek');
    REEKS specs[2] $14 _temporary_ (
        'Cardiologie' 'Oncologie');

    /* Verborgen biases per instelling en per specialisme */
    REEKS f_bias[3] _temporary_ (1.5 0.0 -1.5);
    REEKS s_bias[2] _temporary_ (1.0 -1.0);

    DOE enc = 1 TOT 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));
        facility  = facs[fi];
        specialty = specs[si];

        /* Echte interactieterm instelling x specialisme */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Blijf op een patiëntervaringsschaal van 1-10 */
        ALS satisfaction > 10 DAN satisfaction = 10;
        ALS satisfaction < 1  DAN satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        UITVOER;
    EINDE;

    BEWAREN facility specialty satisfaction;
UITVOEREN;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Stap 2 - Bekijk de ruwe scoreverdeling

Bevestig vóór het modelleren dat de synthetische scores zich goed gedragen
en bekijk de gemiddelden per cel instelling x specialisme. De
percentiel-sleutelwoorden van `PROC MEANS` (`P25`, `MEDIAN`, `P75`) vatten de
totale spreiding samen; de tweede aanroep toont de celgemiddelden die de
interactie dragen die de factorisatiemachine zal proberen terug te vinden.


In [2]:
PROCEDURE GEMIDDELDEN GEGEVENS=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    VARIABELE satisfaction;
    label satisfaction='Tevredenheid';
UITVOEREN;

PROCEDURE GEMIDDELDEN GEGEVENS=encounters mean NWAY maxdec=2;
    KLASSE facility specialty;
    VARIABELE satisfaction;
    label facility='Instelling' specialty='Specialisme' satisfaction='Tevredenheid';
UITVOEREN;


                                                  The MEANS Procedure

 Variable      Label                N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ----------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Tevredenheid       100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 ----------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                     Analysis Variable : satisfaction Tevredenheid

                                                                      N
                                     Instelling      Specialisme    Obs       Mean
                                     ---------------------------------------------
       


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Stap 3 - Fit de factorisatiemachine

We modelleren `satisfaction` als het interval-**doel** en de twee categorische
velden `facility` en `specialty` als nominale **invoer**-kenmerken.
Belangrijkste opties:

- `LEARN=0.02` - de stapgrootte van de stochastische gradiëntafdaling. Bij
  dit kleine, gestandaardiseerde ontwerp houdt een bescheiden snelheid de
  optimalisator stabiel terwijl de celstructuur toch wordt gefit.
- `L2=0.0005` - lichte L2-regularisatie, genoeg om te voorkomen dat de
  factoren te veel krimpen richting het algemene gemiddelde.
- `SEED=20240531` - reproduceerbare initialisatie.
- `OUT=scored` - schrijft voorspellingen per rij (`P_satisfaction`).
- `OUTSTAT=fitstats` - legt de RMSE-geschiedenis per iteratie vast zodat we
  convergentie kunnen bevestigen.

De `ID`-instructie kopieert de sleutelvelden naar de gescoorde uitvoer zodat
elke voorspelling gelabeld blijft met zijn instelling en specialisme.


In [3]:
PROCEDURE factmac GEGEVENS=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    INVOER facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
UITVOEREN;



                         The FACTMAC Procedure

  Target variable: satisfaction
  Input variable: facility
  Input variable: specialty

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Stap 4 - Bevestig convergentie

De `OUTSTAT=`-tabel legt de trainings-RMSE bij elke SGD-iteratie vast. Een
monotoon dalende curve die afvlakt vertelt ons dat het model is geconvergeerd
binnen het (standaard 100) iteratiebudget.


In [4]:
PROCEDURE AFDRUKKEN GEGEVENS=fitstats(obs=15) label noobs;
UITVOEREN;



ITERATION          RMSE
---------  ------------
1          1.6784734207
2          1.2915242083
3          1.2679925124
4          1.2654232966
5          1.2650259551
6          1.2649577538
7          1.2649457032
8          1.2649435534
9          1.2649431684
10         1.2649430993
11         1.2649430869
12         1.2649430847
13         1.2649430843
14         1.2649430842
15         1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Stap 5 - Evalueer de reconstructiefout

De gescoorde tabel draagt de geobserveerde `satisfaction` naast de
`P_satisfaction` van het model. We leiden het residu en de absolute fout af
en vatten ze samen. Een residu dat rond nul is gecentreerd en een spreiding
van voorspelde scores die de geobserveerde spreiding benadert (in plaats van
in te storten tot een platte lijn op het algemene gemiddelde) wijzen erop dat
de factorisatiemachine de ingebedde structuur instelling x specialisme
reproduceert.


In [5]:
GEGEVENS resid;
    INSTELLEN scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=scored(obs=10) label noobs;
    label facility='Instelling' specialty='Specialisme' satisfaction='Tevredenheid'
          p_satisfaction='Voorspelde tevredenheid';
UITVOEREN;

PROCEDURE GEMIDDELDEN GEGEVENS=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    VARIABELE satisfaction p_satisfaction residual abs_err;
    label satisfaction='Tevredenheid' p_satisfaction='Voorspelde tevredenheid'
          residual='Residu' abs_err='Absolute fout';
UITVOEREN;


  Instelling  Specialisme  Tevredenheid  Voorspelde tevredenheid
------------  -----------  ------------  -----------------------
Zuidkliniek   Oncologie             6.3             6.1577265381
Noordkliniek  Oncologie             5.4             6.0430846516
CentraalMed   Cardiologie           7.9             9.5419769923
Zuidkliniek   Cardiologie           4.7             5.8019161993
CentraalMed   Oncologie             6.2             5.9284427651
Noordkliniek  Cardiologie            10             7.6719465958
Noordkliniek  Oncologie             5.9             6.0430846516
Noordkliniek  Cardiologie            10             7.6719465958
Zuidkliniek   Cardiologie           4.9             5.8019161993
CentraalMed   Oncologie             6.2             5.9284427651

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label                           N        Mean     Std Dev     Minimum   Lower Quartil


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Stap 6 - Breng de prestaties per instelling x specialisme naar boven

Voor kwaliteitsverbeteringsteams is de bruikbare weergave de voorspelde
score per **combinatie instelling x specialisme**. Combinaties waarvan de
door het model voorspelde ervaring ruim onder het systeemgemiddelde ligt,
zijn kandidaten voor beoordeling; de kolom absolute fout toont waar het
model netjes past en waar het afgekapte plafond van 1-10 beperkt hoe hoog
een lineaire factorisatiemachine kan reiken.


In [6]:
PROCEDURE GEMIDDELDEN GEGEVENS=resid mean NWAY maxdec=3;
    KLASSE facility specialty;
    VARIABELE p_satisfaction abs_err;
    label facility='Instelling' specialty='Specialisme' p_satisfaction='Voorspelde tevredenheid'
          abs_err='Absolute fout';
UITVOEREN;


                                                  The MEANS Procedure

                                     Analysis Variable : p_satisfaction Voorspelde tevredenheid

                                                                      N
                                     Instelling      Specialisme    Obs       Mean
                                     ---------------------------------------------
                                     CentraalMed     Cardiologie     13      9.542

                                                     Oncologie       20      5.928

                                     Noordkliniek    Cardiologie     18      7.672

                                                     Oncologie       16      6.043

                                     Zuidkliniek     Cardiologie     20      5.802

                                                     Oncologie       13      6.158
                                     ---------------------------------------------

       


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## De resultaten interpreteren

De iteratiegeschiedenis uit `OUTSTAT=` toont de trainings-RMSE dalend van
ongeveer 1,68 bij de eerste doorloop naar een plateau nabij **1,265** rond de
zevende iteratie, wat bevestigt dat de SGD-fit ruim binnen het
iteratiebudget goed is geconvergeerd. De Fit Statistics rapporteren een
**trainings-R-kwadraat van 0,516**, een **gemiddelde absolute fout van
0,954** scorepunten en een **RASE van 1,25** — de factorisatiemachine
verklaart ongeveer de helft van de variantie in een tevredenheidsscore van
1-10 met een standaarddeviatie van 1,81, dus leert het echt structuur in
plaats van het algemene gemiddelde te voorspellen. De residusamenvatting
bevestigt dit: het gemiddelde residu is vrijwel nul (0,020) en de voorspelde
scores lopen van 5,80 tot 9,54 (standaarddeviatie 1,27), wat de geobserveerde
spreiding volgt — zonder die volledig te evenaren.

De tabel instelling x specialisme maakt van het latente model iets waarmee
een zorgkwaliteitsteam aan de slag kan. Het model rangschikt
`CentraalMed`/`Cardiologie` het hoogst (voorspelling 9,54) en
`Zuidkliniek`/`Cardiologie` het laagst (voorspelling 5,80), waarbij de
ingebedde interactie wordt teruggevonden waarin Cardiologie op sommige
locaties uitstekend is en op andere zwak. De kolom absolute fout is eerlijk
over de beperkingen van het model: de twee Oncologie-cellen worden bijna
exact gereproduceerd (gemiddelde absolute fout 0,19-0,24), terwijl
`Noordkliniek`/`Cardiologie` wordt onderschat (fout 2,33) omdat de werkelijke
scores zich opstapelen tegen het afgekapte plafond van 1-10 dat een lineaire
factorisatiemachine niet kan bereiken.

**Volgende stappen** die een practicus zou kunnen nemen: een
`PARTITION`-houdout toevoegen om overfitting tegen te gaan, `LEARN=` en
`L2=` afstemmen om bias tegen variantie af te wegen, of de kenmerkenset
uitbreiden (zorgverlener, bezoektype, seizoen) zodat de factorisatiemachine
hogere-orde ervaringsdrijfveren kan modelleren. Op een volledig
gelicentieerde installatie zou een groter raster instelling x specialisme
met meer contacten per cel het model in staat stellen fijnere
interactiestructuur op te lossen dan het zescellige ontwerp dat hier wordt
getoond.
